# Step 4.1: Architectural Strategy and Environment Setup

## 1. Revised Augmentation Strategy Note

This notebook succeeds `3_cm_no_augs.ipynb` and corrects an overcorrection: the previous iteration removed all spatial augmentation entirely, but in doing so it under-regularised the model. This version reintroduces a very light augmentation pipeline (small random flips, rotations, translations, and zooms) combined with a weakened Mixup (alpha reduced to 0.2), a slightly higher dropout rate (0.4), and a slightly higher weight decay on the optimiser. The goal is to recover the regularisation signal without reintroducing the excessive noise that harmed earlier runs.

In [1]:
import os
# python standard library imports
from pathlib import Path
import json
import math
# model building imports
from keras import Model, layers, Sequential
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
# other imports
from keras.utils import image_dataset_from_directory

## 2. Hardware Optimisation

Implements `set_memory_growth` to prevent TensorFlow from allocating the entirety of the VRAM at startup, avoiding hard crashes during execution, and enables Accelerated Linear Algebra (XLA) Just-In-Time compilation to fuse TensorFlow operations into optimised GPU kernels, significantly accelerating batch processing after the initial compilation overhead.

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")


# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
tf.config.optimizer.set_jit(True)
print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']
XLA JIT enabled.


# Step 4.2: Custom Convolutional Neural Network Design

## 1. Residual Block Implementation

Defines a custom `ResidualBlock` by subclassing `keras.layers.Layer`, ensuring Keras correctly tracks all internal weights and biases. This version extends the block to a proper two-convolution residual design: the first convolution applies the specified kernel and stride, followed by Batch Normalisation and activation, and a second 3×3 convolution with stride 1 refines the features before the shortcut merge. He normal initialisation is used throughout for stable gradient flow in ReLU networks. The projection shortcut is only constructed when actually needed (when strides differ or filter counts change) avoiding unnecessary parameters.

In [3]:
class ResidualBlock(layers.Layer):
    """
    Single residual block: Conv → BN → Activation + shortcut projection.

    Storing conv/bn/activation as named attributes of a Layer subclass
    guarantees Keras tracks their weights correctly.
    The original bug stored these inside plain Python dicts inside a plain
    Python list — Keras never registered them, so they were never trained.
    """

    def __init__(self, in_filters, filters, kernel_size, stride, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size
        self.stride      = stride
        self.activation  = activation

        # He normal: correct initialisation for ReLU networks
        init = "he_normal"

        self.needs_projection = (stride != 1) or (in_filters != filters)  # <-- computed once, stored

        self.conv1     = layers.Conv2D(filters, kernel_size, strides=stride,
                                      padding="same", use_bias=False,
                                      kernel_initializer=init)
        self.bn1       = layers.BatchNormalization(momentum=0.9)
        self.actv1     = layers.Activation(activation)

        self.conv2     = layers.Conv2D(filters, (3, 3), strides=1,
                                      padding="same", use_bias=False,
                                      kernel_initializer=init)
        self.bn2       = layers.BatchNormalization(momentum=0.9)

        # Only build these layers if they'll actually be used
        if self.needs_projection:
            self.shortcut_conv = layers.Conv2D(filters, (1, 1), strides=stride,
                                               padding="same", use_bias=False,
                                               kernel_initializer=init)
            self.shortcut_bn   = layers.BatchNormalization(momentum=0.9)

        self.add       = layers.Add()
        self.actv2 = layers.Activation(activation)

    def call(self, x, training=False):
        skip = x
        if self.needs_projection:
            skip = self.shortcut_conv(skip)
            skip = self.shortcut_bn(skip, training=training)
    
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = self.actv1(x)         # activation between the two convs
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.add([x, skip])
        return self.actv2(x)      # activation after the merge

    def get_config(self):
        return {**super().get_config(),
                "filters": self.filters, "kernel_size": self.kernel_size,
                "stride": self.stride,   "activation": self.activation}

## 2. Main CNN Architecture (`MyCNN`)

Integrates a `layers.Rescaling(1./255)` layer directly into the model definition to ensure compatibility with transfer learning pipelines and standardise input tensors. The constructor now explicitly tracks the input filter depth (`in_f`) across residual blocks so that each block receives the correct number of input channels. The dense classification head uses a dropout rate of 0.3 by default, reduced from the 0.5 used in earlier iterations, as regularisation is now distributed across augmentation, Mixup, and weight decay.

In [4]:
class MyCNN(Model):
    def __init__(self, conv_configs, dense_configs, num_classes, augmentation_layer=None, activation="relu", dropout_rate=0.3, **kwargs):
        super().__init__(**kwargs, name="my_cnn")
        self.num_classes = num_classes
        self.conv_configs = conv_configs
        self.dense_configs = dense_configs
        self.augmentation_layer = augmentation_layer
        self.activation = activation
        self.dropout_rate = dropout_rate

        # 1. ADD RESCALING HERE (The fix for your Transfer Learning compatibility)
        self.rescaling = layers.Rescaling(1./255)

        # Store as a Python list of Layer objects assigned to self.
        # Keras DOES track a list of Layers set as an attribute via __setattr__,
        # as long as the list itself is set at attribute assignment time (not grown later).
        # Safest pattern: build the full list first, then assign once.
        blocks_list = []
        in_f = 3  # RGB input — or 1 if grayscale
        for i, (f, k, s) in enumerate(conv_configs):
            blocks_list.append(
                ResidualBlock(in_f, f, k, s, activation=activation, name=f"block_{i}")
            )
            in_f = f  # output of this block becomes input of the next
        self.blocks=blocks_list


        self.gap = layers.GlobalAveragePooling2D(name="GAP")
        dense_list = []
        for i, u in enumerate(self.dense_configs):
            dense_list.append(layers.Dense(u, activation=self.activation, name=f"fc_{i}"))
            dense_list.append(layers.Dropout(self.dropout_rate, name=f"drop_{i}"))
        self.dense_layers = dense_list
        self.classifier = layers.Dense(self.num_classes, activation='softmax', name="head")

    def get_config(self):
        # Obtain the base configuration from the superclass
        config = super().get_config()
        # Add the custom arguments to the dictionary
        config.update({
            "num_classes": self.num_classes,
            "conv_configs": self.conv_configs,
            "dense_configs": self.dense_configs,
            "augmentation_layer": self.augmentation_layer,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        x = self.rescaling(inputs)
        if self.augmentation_layer is not None:
            x = self.augmentation_layer(x, training=training)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.gap(x)
        for layer in self.dense_layers:
            # Dropout needs training flag; Dense does not
            x = layer(x, training=training) if isinstance(layer, layers.Dropout) else layer(x)
        return self.classifier(x)

# Step 4.3: Hyperparameters and Data Pipeline

## 1. Global Configuration

Establishes critical training parameters, notably reducing the image resolution to 320×320 and the batch size to 8 to accommodate the model's two-convolution residual blocks within the 8GB VRAM constraint. The learning rate is set to 5e-4, adjusted proportionally to the reduced batch size relative to earlier runs. Dynamically generates `Checkpoints` and `Metrics` directories to store model artefacts and training logs safely.

In [7]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
IMAGE_SIZE     = (320, 320)
BATCH_SIZE     = 8
EPOCHS         = 64       # good balance
LEARNING_RATE  = 5e-4
N_CLASSES      = 23

data_dir_path = Path("..\wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

## 2. Dataset Instantiation

Re-instantiates the `tf.data.Dataset` objects from the partitioned directories, applying the new 320×320 resolution and the adjusted batch size whilst maintaining categorical label encoding.

In [8]:
# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


# Step 4.4: Regularisation and Architecture Configuration

## 1. Augmentation Pipeline and Mixup Implementation

Defines a lightweight `Sequential` augmentation block (`cnn_augmentation`) applied inside the model at training time, comprising small random horizontal flips, rotations (±8%), translations (±8%), and zooms (±8%). These magnitudes are intentionally faint — sufficient to act as a regulariser but not so aggressive as to obscure the fine-grained stylistic features that distinguish WikiArt classes.

Also defines a custom `mixup` function with a reduced alpha of 0.2, producing softer label blends than the previous 0.4. The training pipeline caches the raw data first, shuffles with a buffer of 1000, then applies Mixup dynamically per epoch via `AUTOTUNE`-parallelised mapping, followed by prefetching.

In [9]:
cnn_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomZoom(0.08),
], name="cnn_aug")

# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
# Forces the model to learn smoother decision boundaries rather than
# memorising exact compositions — especially useful for fine-grained style tasks.
def mixup(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 1.0-alpha, 1.0)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

train_ds_mixed = (
    train_ds
    .cache() # Original data cache
    .shuffle(1000)
    .map(mixup, num_parallel_calls=AUTOTUNE) # Dinamic mixup per epoch
    .prefetch(AUTOTUNE)
)


val_ds  = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## 2. Architecture Configuration

Defines the specific filter counts, kernel sizes, and strides for the residual blocks (`conv_setup`), alongside the dense layer dimensions (`dense_setup`), reduced to `[512, 256]` from the previous `[1024, 512]` to limit overfitting in the classification head. Loads the previously serialised class weights from a JSON file to address the significant dataset imbalance during the loss calculation.

In [10]:
# ── Custom CNN architecture config ───────────────────────────────────────────
conv_setup = [
    (64,  (7, 7), 2),
    (64,  (3, 3), 1),
    (128, (3, 3), 2),
    (128, (3, 3), 1),
    (256, (3, 3), 2),
    (256, (3, 3), 1),
    (512, (3, 3), 2),
    (512, (3, 3), 1),
]
dense_setup = [512, 256]

# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


# Step 4.5: Model Compilation and Lifecycle Management

## 1. Model Instantiation

Instantiates `MyCNN` with the light augmentation layer attached, the revised architecture configuration, a dropout rate of 0.4, and 23 output classes.

In [11]:
model = MyCNN(
    augmentation_layer=cnn_augmentation,
    conv_configs=conv_setup,
    dense_configs=dense_setup,
    dropout_rate=0.4,
    num_classes=N_CLASSES,
)

## 2. Metrics and Loss Formulation

Instantiates fresh, stateful metrics for the model, featuring the Macro **F1-score** (`tfa.metrics.F1Score`) alongside Categorical Accuracy and AUC to ensure a balanced evaluation across all 23 classes. Compiles the model using `CategoricalCrossentropy` with a `label_smoothing` factor of 0.1 to penalise overconfidence in predictions. Replaces SGD with `tfa.optimizers.AdamW` using a weight decay of 1e-4, providing adaptive gradient updates alongside explicit L2-style regularisation, a more suitable optimiser for a model that already applies several other regularisation mechanisms.

In [12]:
def make_metrics(num_classes):
    """Fresh metric instances per model — metrics are stateful and must not be shared."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]

In [13]:
# Compile the model
model.compile(
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1), 
    optimizer=tfa.optimizers.AdamW(learning_rate=LEARNING_RATE, name="optimizer", weight_decay=1e-4), 
    metrics=make_metrics(num_classes=N_CLASSES)
)

## 3. Callback Configuration

Implements a custom cosine annealing schedule with a linear warmup phase to prevent outsized gradient updates during the initial epochs. Configures an `EarlyStopping` callback monitoring validation loss with a patience of 10 epochs (increased from 7), ensuring the best weights are automatically restored whilst allowing the model more time to recover from temporary plateaus introduced by the augmentation noise.

In [14]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Linear warmup then cosine annealing.

    Warmup matters especially for the larger LR used with MyCNN (2e-3):
    without it, the first few batches produce outsized gradient updates.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler


In [15]:
# Define Callbacks
checkpoint_callback = ModelCheckpoint(
    checkpoints_folder_path / f"checkpoint_{model.name}",
    save_best_only=True,
    monitor="val_loss",
    verbose=0
)
metrics_callback = CSVLogger(metrics_folder_path / f"metric_{model.name}.csv")

In [16]:
lr_scheduler_callback = LearningRateScheduler(make_cosine_warmup_scheduler(LEARNING_RATE, EPOCHS, warmup_epochs=3))

In [17]:
# EarlyStopping: stops training if val_loss doesn't improve for "patience" epochs
# and restores the best weights automatically
early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)


In [18]:
callbacks = [
    checkpoint_callback,
    metrics_callback,
    lr_scheduler_callback,
    early_stopping_callback
]

# Step 4.6: Execution and Simple Evaluation

## 1. Training Loop

Executes the `model.fit()` routine using the Mixup-augmented training dataset, passing the pre-calculated class weights to handle the significant dataset imbalance.

*Utilises `clear_session()` immediately after training and evaluation to flush the GPU memory and prevent state leakage into subsequent model experiments.*

In [19]:
# Train the model
model_fit_data = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)
model_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
clear_session()

model_fit_data, model_eval_data

Epoch 1/64
1166/1166 [==============================] - ETA: 0s - loss: 3.1084 - accuracy: 0.1123 - auc: 0.5583 - f1_score: 0.0903WARNING:tensorflow:Using a while_loop for converting RngReadAndSkip cause there is no registered converter for this op.


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


1166/1166 [==============================] - 361s 241ms/step - loss: 3.1084 - accuracy: 0.1123 - auc: 0.5583 - f1_score: 0.0903 - val_loss: 2.8993 - val_accuracy: 0.1421 - val_auc: 0.7187 - val_f1_score: 0.1149 - lr: 1.6667e-04
Epoch 2/64
1166/1166 [==============================] - ETA: 0s - loss: 2.9959 - accuracy: 0.1517 - auc: 0.5895 - f1_score: 0.1140WARNING:tensorflow:Using a while_loop for converting RngReadAndSkip cause there is no registered converter for this op.


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


...

1166/1166 [==============================] - 280s 240ms/step - loss: 2.2518 - accuracy: 0.4563 - auc: 0.7064 - f1_score: 0.3923 - val_loss: 2.1064 - val_accuracy: 0.4393 - val_auc: 0.9048 - val_f1_score: 0.4092 - lr: 1.9258e-04
Epoch 40/64
1166/1166 [==============================] - 254s 218ms/step - loss: 2.2401 - accuracy: 0.4587 - auc: 0.7071 - f1_score: 0.3955 - val_loss: 2.1924 - val_accuracy: 0.4021 - val_auc: 0.8964 - val_f1_score: 0.3644 - lr: 1.8013e-04
Epoch 41/64
1166/1166 [==============================] - 253s 217ms/step - loss: 2.2366 - accuracy: 0.4624 - auc: 0.7062 - f1_score: 0.4004 - val_loss: 2.2827 - val_accuracy: 0.3775 - val_auc: 0.8911 - val_f1_score: 0.3363 - lr: 1.6786e-04
Epoch 42/64
1166/1166 [==============================] - 260s 223ms/step - loss: 2.2337 - accuracy: 0.4623 - auc: 0.7075 - f1_score: 0.3988 - val_loss: 2.1345 - val_accuracy: 0.4322 - val_auc: 0.9087 - val_f1_score: 0.3987 - lr: 1.5582e-04
Epoch 43/64
1166/1166 [=============================

INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


1166/1166 [==============================] - 276s 236ms/step - loss: 2.2196 - accuracy: 0.4677 - auc: 0.7065 - f1_score: 0.4054 - val_loss: 2.0595 - val_accuracy: 0.4588 - val_auc: 0.9147 - val_f1_score: 0.4237 - lr: 1.2130e-04
Epoch 46/64
1166/1166 [==============================] - 255s 218ms/step - loss: 2.2171 - accuracy: 0.4728 - auc: 0.7043 - f1_score: 0.4115 - val_loss: 2.3187 - val_accuracy: 0.3635 - val_auc: 0.8840 - val_f1_score: 0.3527 - lr: 1.1044e-04
Epoch 47/64
1166/1166 [==============================] - 253s 217ms/step - loss: 2.2292 - accuracy: 0.4652 - auc: 0.7038 - f1_score: 0.4013 - val_loss: 2.1480 - val_accuracy: 0.4342 - val_auc: 0.9020 - val_f1_score: 0.4111 - lr: 9.9946e-05
Epoch 48/64
1166/1166 [==============================] - 257s 221ms/step - loss: 2.2219 - accuracy: 0.4716 - auc: 0.7062 - f1_score: 0.4096 - val_loss: 2.1794 - val_accuracy: 0.4167 - val_auc: 0.9016 - val_f1_score: 0.3754 - lr: 8.9852e-05
Epoch 49/64
1166/1166 [=============================

(<keras.callbacks.History at 0x21cab59bd30>,
 {'loss': 2.036302328109741,
  'accuracy': 0.4698318541049957,
  'auc': 0.9191275238990784,
  'f1_score': 0.4326980412006378})

## 2. Performance Evaluation

Evaluates the restored best weights against the unseen `test_ds`, returning a dictionary of metrics to benchmark the model's generalisation capabilities.

In [20]:
model_eval_data

{'loss': 2.036302328109741,
 'accuracy': 0.4698318541049957,
 'auc': 0.9191275238990784,
 'f1_score': 0.4326980412006378}